# Huấn luyện và Đánh giá CAE trên Google Colab

**Đề tài:** Xây dựng và đánh giá mạng Convolutional Autoencoder trong phát hiện và khoanh vùng bất thường bề mặt sản phẩm công nghiệp.

> **Nguyên tắc:** Toàn bộ logic huấn luyện, hiệu chỉnh ngưỡng và đánh giá nằm trong các script `.py`. Notebook này chỉ thực hiện kết nối Drive, chuẩn bị môi trường và gọi các lệnh thực thi.

## 1. Kết nối Google Drive & Kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!python -c "import torch; print('PyTorch:', torch.__version__, '| CUDA Available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

## 2. Đồng bộ mã nguồn từ GitHub

*(Nếu là lần đầu tiên, hãy clone repository. Các lần sau chỉ cần `git pull` để cập nhật code mới nhất từ máy cá nhân)*

In [ ]:
import os

# THAY THẾ URL REPOSITORY CỦA BẠN TẠI ĐÂY:
REPO_URL = "https://github.com/<your_username>/industrial-anomaly-cae.git"
REPO_DIR = "/content/industrial-anomaly-cae"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

!pip install -r requirements.txt

## 3. Chuẩn bị Dữ liệu trên Colab Runtime

Sao chép danh mục `bottle` từ Google Drive sang ổ đĩa tạm `/content/` để tăng tốc độ nạp dữ liệu trong quá trình huấn luyện.

In [ ]:
# Tạo thư mục tạm trên Colab runtime
!mkdir -p /content/cae_data/mvtec_anomaly_detection

# Đường dẫn lưu dataset trên Google Drive của bạn
DRIVE_DATASET_PATH = "/content/drive/MyDrive/CAE_Project/datasets/mvtec_anomaly_detection/bottle"
LOCAL_DATASET_PATH = "/content/cae_data/mvtec_anomaly_detection/bottle"

# Sao chép danh mục sang /content/
!cp -r "{DRIVE_DATASET_PATH}" "/content/cae_data/mvtec_anomaly_detection/"
!ls -la "{LOCAL_DATASET_PATH}"

## 4. Huấn luyện Mô hình CAE (train.py)

Lưu checkpoint, log loss và đồ thị trực tiếp vào Google Drive để tránh mất dữ liệu khi ngắt phiên Colab.

In [ ]:
!python train.py \
  --config configs/bottle.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E1_bottle_mse_seed42

## 5. Hiệu chỉnh Ngưỡng trên Validation (calibrate.py)

Xác định `image_threshold` và `pixel_threshold` dựa hoàn toàn trên ảnh bình thường của tập validation.

In [ ]:
!python calibrate.py \
  --config configs/bottle.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E1_bottle_mse_seed42

## 6. Đánh giá Độc lập trên Tập Test (evaluate.py)

Tính AUROC, F1-score, Precision, Recall, Pixel AUROC, IoU, Dice và trực quan hóa các ca TP, TN, FP, FN.

In [ ]:
!python evaluate.py \
  --config configs/bottle.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E1_bottle_mse_seed42